---
id: "readme"
template: "markdown"
name: "readme"
position:
  x: 0
  y: -40
dimensions:
  width: 900
  height: 360
config:
  md: |-
    # m_LOAD_CUSTOMER_CLEAN (converted from Informatica PowerCenter)
    Folder `RETAIL_DWH`, workflow `wf_LOAD_CUSTOMER_CLEAN`, session `s_m_LOAD_CUSTOMER_CLEAN`.

    | Informatica | Lakeflow Designer |
    |---|---|
    | SQ_CUSTOMERS | `src_customers` |
    | EXP_STANDARDIZE | `exp_standardize` (Transform) |
    | FIL_ACTIVE | `fil_active` (Filter) |
    | T_CUSTOMER_CLEAN (truncate + load) | `tgt_map` + `out_customer_clean` (overwrite) |

    `IS_DATE` / `TO_DATE` become `TRY_TO_TIMESTAMP` (ANSI-safe). `||` keeps Informatica NULL-as-empty semantics via `COALESCE`.
---

In [0]:
"""
id: "src_customers"
template: "source"
templateVersion: "1.0.0"
name: "src_customers"
position:
  x: 0
  y: 360
description:
  text: "SQ_CUSTOMERS: Oracle ERP.CUSTOMERS (no SQL override, no source filter)."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_customers"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_customers"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_customers.data"] = out["data"]


In [0]:
"""
id: "exp_standardize"
template: "transform"
templateVersion: "1.0.0"
name: "exp_standardize"
position:
  x: 300
  y: 360
description:
  text: "EXP_STANDARDIZE: trim / case / default the attributes; IS_DATE + TO_DATE for two signup formats."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "*"
    - "INITCAP(TRIM(first_name)) AS o_first_name"
    - "INITCAP(TRIM(last_name)) AS o_last_name"
    - "COALESCE(INITCAP(TRIM(first_name)), '') || ' ' || COALESCE(INITCAP(TRIM(last_name)), '') AS o_full_name"
    - "LOWER(TRIM(email)) AS o_email"
    - "UPPER(TRIM(country)) AS o_country"
    - "CASE WHEN segment IS NULL OR TRIM(segment) = '' THEN 'UNKNOWN' ELSE UPPER(TRIM(segment)) END AS o_segment"
    - "CASE WHEN TRY_TO_TIMESTAMP(signup_date, 'dd.MM.yyyy') IS NOT NULL THEN CAST(TRY_TO_TIMESTAMP(signup_date, 'dd.MM.yyyy') AS DATE) ELSE CAST(TRY_TO_TIMESTAMP(signup_date, 'yyyy-MM-dd') AS DATE) END AS o_signup_date"
input:
  - node: "src_customers"
    input_port: "data"
    output_port: "data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "*",
        "INITCAP(TRIM(first_name)) AS o_first_name",
        "INITCAP(TRIM(last_name)) AS o_last_name",
        "COALESCE(INITCAP(TRIM(first_name)), '') || ' ' || COALESCE(INITCAP(TRIM(last_name)), '') AS o_full_name",
        "LOWER(TRIM(email)) AS o_email",
        "UPPER(TRIM(country)) AS o_country",
        "CASE WHEN segment IS NULL OR TRIM(segment) = '' THEN 'UNKNOWN' ELSE UPPER(TRIM(segment)) END AS o_segment",
        "CASE WHEN TRY_TO_TIMESTAMP(signup_date, 'dd.MM.yyyy') IS NOT NULL THEN CAST(TRY_TO_TIMESTAMP(signup_date, 'dd.MM.yyyy') AS DATE) ELSE CAST(TRY_TO_TIMESTAMP(signup_date, 'yyyy-MM-dd') AS DATE) END AS o_signup_date"
    ]
}
inputs = {
    "data": ctx["src_customers.data"]
}
out = run(config, inputs, spark)
ctx["exp_standardize.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "fil_active"
template: "filter"
templateVersion: "1.0.0"
name: "fil_active"
position:
  x: 600
  y: 360
description:
  text: "FIL_ACTIVE: STATUS = 'A'."
  hash: ""
previewMode: "1000"
config:
  condition: "status = 'A'"
input:
  - node: "exp_standardize"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "status = 'A'"
}
inputs = {
    "data": ctx["exp_standardize.transformed_data"]
}
out = run(config, inputs, spark)
ctx["fil_active.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "tgt_map"
template: "transform"
templateVersion: "1.0.0"
name: "tgt_map"
position:
  x: 900
  y: 360
description:
  text: "Connector map FIL_ACTIVE -> T_CUSTOMER_CLEAN target ports."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "customer_id"
    - "o_full_name AS full_name"
    - "o_email AS email"
    - "o_country AS country"
    - "o_segment AS segment"
    - "o_signup_date AS signup_date"
input:
  - node: "fil_active"
    input_port: "data"
    output_port: "filtered_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "customer_id",
        "o_full_name AS full_name",
        "o_email AS email",
        "o_country AS country",
        "o_segment AS segment",
        "o_signup_date AS signup_date"
    ]
}
inputs = {
    "data": ctx["fil_active.filtered_data"]
}
out = run(config, inputs, spark)
ctx["tgt_map.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "out_customer_clean"
template: "output"
templateVersion: "1.0.0"
name: "out_customer_clean"
position:
  x: 1200
  y: 360
description:
  text: "T_CUSTOMER_CLEAN (Truncate target table option = YES -> overwrite)."
  hash: ""
previewMode: "full"
config:
  catalog: "daphnedemos_catalog"
  schema: "migration_skill_tests"
  table_name: "infa_01_customer_clean"
input:
  - node: "tgt_map"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "daphnedemos_catalog",
    "schema": "migration_skill_tests",
    "table_name": "infa_01_customer_clean"
}
inputs = {
    "data": ctx["tgt_map.transformed_data"]
}
out = run(config, inputs, spark)
